# Estonian Energy Resilience Under Disruption
**42578 Advanced Business Analytics — DTU, Spring 2026**

## Introduction

In January 2026, Estonia's electricity system faced an acute stress test. Monthly average prices reached ~154 €/MWh with hourly peaks near 505 €/MWh, driven by unusually cold weather, weak wind generation, and production outages. The February maximum rose further to 655 €/MWh. Beyond prices, the episode exposed a deeper structural question: how dependent is Estonia's supply security on cross-border imports, and what would happen if those imports were unavailable?

This project analyses January 2026 as a resilience problem. On the supply side, a spatio-temporal graph neural network (ST-GNN) models Estonian electricity production under four scenarios — full grid, full isolation, and two levels of wind expansion. On the demand side, a SARIMAX model with Monte Carlo simulation estimates hourly consumption uncertainty. Combining both gives a probabilistic picture of surplus and deficit hours under each scenario.

The central policy question mirrors a public claim made by Utilitas Wind in February 2026: that additional offshore wind capacity could have cut January prices roughly in half. We evaluate that claim using weather-grounded counterfactual simulations on the actual crisis month.

**References:** ERR News (Jan–Feb 2026); Konkurentsiamet *Energiaturgude ülevaade veebruar 2026* (Apr 2026); Ärileht/Delfi, Utilitas Wind statement (Feb 2026).

## Data Sources

| Dataset | Source | Coverage |
|---------|--------|----------|
| Electricity prices, cross-border flows, system production | Elering Dashboard API | 2019–Feb 2026, hourly |
| Wind speed (100 m), temperature, pressure | Open-Meteo ERA5 reanalysis | 2019–2026, hourly |
| Generation by type (wind, oil shale, biomass…) | ENTSOE Transparency Platform | 2019–2026, hourly |
| Wind farm locations & planning status | Estonian Land Board GIS | Static (2026) |

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

DATA    = '../data'
FIGURES = '../figures'
os.makedirs(FIGURES, exist_ok=True)

SUPPLY_CSV      = f'{DATA}/gnn_supply_scenarios_jan2026.csv'
TOTAL_AVAIL_CSV = f'{DATA}/total_available_energy_jan2026.csv'
DEMAND_CSV      = f'{DATA}/demand_mc_jan2026.csv'
WIND_CSV        = f'{DATA}/wind_production_scenarios.csv'

print('Setup complete')

---
## Crisis Context — January 2026

Estonia's grid is normally balanced by domestic generation plus net imports from Finland and Latvia. The plots below show that balance and how large the import buffer actually was during January 2026.

In [ ]:
try:
    avail  = pd.read_csv(TOTAL_AVAIL_CSV, parse_dates=['timestamp'], index_col='timestamp')
    demand = pd.read_csv(DEMAND_CSV,      parse_dates=['timestamp'], index_col='timestamp')
    if avail.index.tz  is None: avail.index  = avail.index.tz_localize('UTC')
    if demand.index.tz is None: demand.index = demand.index.tz_localize('UTC')
    common = avail.index.intersection(demand.index)
    idx = common.to_numpy()

    # Net imports = S0 connected (prod+imports) minus S0 isolated (prod only)
    net_imports = avail.loc[common, 'supply_s0_p50'].values - avail.loc[common, 'supply_s0iso_p50'].values

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

    axes[0].fill_between(idx, avail.loc[common, 'supply_s0_p10'].values,
                         avail.loc[common, 'supply_s0_p90'].values,
                         alpha=0.15, color='green', label='S0 connected P10–P90')
    axes[0].plot(idx, avail.loc[common, 'supply_s0_p50'].values,
                 color='green', lw=2, label='S0: Connected (production + imports)')
    axes[0].fill_between(idx, demand.loc[common, 'demand_p5'].values,
                         demand.loc[common, 'demand_p95'].values,
                         alpha=0.15, color='teal', label='Demand P5–P95')
    axes[0].plot(idx, demand.loc[common, 'demand_actual'].values,
                 color='red', lw=1.5, linestyle='--', label='Actual consumption')
    axes[0].set_ylabel('Energy (MW)')
    axes[0].set_title('January 2026 — total available energy vs consumption (connected grid)')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    bar_c = ['steelblue' if v >= 0 else 'crimson' for v in net_imports]
    axes[1].bar(idx, net_imports, color=bar_c, alpha=0.75, width=0.04)
    axes[1].axhline(0, color='black', lw=1)
    axes[1].set_ylabel('Net imports (MW)')
    axes[1].set_title('Cross-border net imports — January 2026 (positive = EE importing)')
    axes[1].tick_params(axis='x', rotation=30); axes[1].grid(alpha=0.3)

    plt.suptitle('Crisis Context — January 2026', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/context_jan2026.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Mean net imports: {net_imports.mean():.0f} MW  "
          f"({net_imports.mean()/demand.loc[common,'demand_actual'].mean()*100:.0f}% of consumption)")
except FileNotFoundError as e:
    print(f'Missing: {e}\nRun available_energy_analysis.ipynb first.')

---
## Demand Analysis — SARIMAX Monte Carlo

Consumption is modelled with SARIMAX(2,0,2)(1,1,2,24) fitted separately on each January 2019–2025, using 24h rolling temperature, wind speed, and a weekend flag as exogenous regressors. 1000 Monte Carlo simulations are generated by drawing a random year's model, randomising its beta coefficients within ±1σ of the cross-year distribution, and adding a residual noise draw. This gives a realistic demand distribution reflecting both model and year-to-year weather uncertainty.

In [ ]:
try:
    demand = pd.read_csv(DEMAND_CSV, parse_dates=['timestamp'], index_col='timestamp')
    if demand.index.tz is None: demand.index = demand.index.tz_localize('UTC')
    idx = demand.index.to_numpy()

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

    axes[0].fill_between(idx, demand['demand_p5'].values, demand['demand_p95'].values,
                         alpha=0.2, color='teal', label='P5–P95 (MC band)')
    axes[0].plot(idx, demand['demand_p50'].values,    color='teal',  lw=2,   label='P50 median')
    axes[0].plot(idx, demand['demand_actual'].values, color='red',   lw=2,   linestyle='--', label='Actual consumption')
    axes[0].plot(idx, demand['demand_mean'].values,   color='navy',  lw=1.2, linestyle=':',  label='MC mean')
    axes[0].set_ylabel('Consumption (MW)')
    axes[0].set_title('SARIMAX Monte Carlo — 1000 simulations, January 2026')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    resid  = demand['demand_actual'].values - demand['demand_p50'].values
    bar_c  = ['crimson' if r < 0 else 'steelblue' for r in resid]
    axes[1].bar(idx, resid, color=bar_c, alpha=0.75, width=0.04)
    axes[1].axhline(0, color='black', lw=1)
    axes[1].set_ylabel('Actual − P50 (MW)')
    axes[1].set_title('Forecast residuals')
    axes[1].tick_params(axis='x', rotation=30); axes[1].grid(alpha=0.3)

    plt.suptitle('Demand Model — January 2026', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/final_demand_mc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Actual mean: {demand['demand_actual'].mean():.0f} MW")
    print(f"P50 mean:    {demand['demand_p50'].mean():.0f} MW")
    print(f"Bias (actual−P50): {resid.mean():.0f} MW")
    print(f"P5–P95 band: {(demand['demand_p95']-demand['demand_p5']).mean():.0f} MW wide")
except FileNotFoundError as e:
    print(f'Missing: {e}\nRun Timeseries.ipynb to generate demand_mc_jan2026.csv')

---
## Wind Scenarios

Counterfactual wind production is estimated using ERA5 reanalysis wind speeds at 100 m, a Vestas V150-4.5 MW power curve, and an air-density correction for January cold conditions (~6% power boost vs standard at sea level). Two scenarios are built on top of the existing 694 MW fleet:

**Scenario A (+323 MW):** Lääneranna area 2 (137 MW) · Pärnu+Tori Põlendmaa (86 MW) · Aidu (100 MW) — all with finalized or near-final permits.

**Scenario B (+887 MW total):** Scenario A plus five pipeline municipalities — Lääneranna pipeline, Tori, Lääne-Nigula, Põhja-Pärnumaa, Lüganuse — in advanced planning with SEA reports published.

In [ ]:
try:
    wind_df = pd.read_csv(WIND_CSV)
    wind_df.index = pd.date_range('2026-01-01', periods=len(wind_df), freq='h', tz='UTC')
    wind_df = wind_df.rename(columns={
        'wind_mwh_baseline': 'Baseline (694 MW)',
        'wind_mwh_scenA':    'Scenario A (+323 MW)',
        'wind_mwh_scenB':    'Scenario B (+887 MW)',
    })
    rated  = {'Baseline (694 MW)': 694, 'Scenario A (+323 MW)': 1017, 'Scenario B (+887 MW)': 1581}
    colors = ['#5a8fc2', '#2ecc71', '#e67e22']
    daily  = wind_df.resample('D').mean()

    fig, axes = plt.subplots(2, 1, figsize=(16, 8))

    for col, clr in zip(wind_df.columns, colors):
        axes[0].plot(wind_df.index.to_numpy(), wind_df[col].values,
                     color=clr, lw=1.4, alpha=0.85, label=col)
    axes[0].set_title('Wind production by scenario — January 2026 (hourly)')
    axes[0].set_ylabel('Production (MW)')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
    axes[0].tick_params(axis='x', rotation=20)

    x = np.arange(len(daily)); w = 0.28
    for i, (col, clr) in enumerate(zip(daily.columns, colors)):
        cf = daily[col] / rated[col] * 100
        axes[1].bar(x + (i-1)*w, cf.values, w, color=clr, alpha=0.8, label=col)
    axes[1].axhline(30.4, color='gray', lw=1.2, linestyle='--', label='Historical avg CF 30.4%')
    axes[1].axhline(15.2, color='red',  lw=1.2, linestyle=':',  label='Jan 2026 avg CF 15.2%')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([d.strftime('%d %b') for d in daily.index], rotation=20)
    axes[1].set_title('Daily capacity factor by scenario')
    axes[1].set_ylabel('Capacity Factor (%)')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, axis='y')

    plt.suptitle('Counterfactual Wind Scenarios — January 2026', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/final_wind_scenarios.png', dpi=150, bbox_inches='tight')
    plt.show()
    for col in wind_df.columns:
        print(f'{col}: mean {wind_df[col].mean():.0f} MW  (CF {wind_df[col].mean()/rated[col]*100:.1f}%)')
except FileNotFoundError as e:
    print(f'Missing: {e}\nRun ursula_wind_counterfactual.ipynb')

---
## GNN Supply Model

The supply model is a spatio-temporal GNN with four nodes (EE, FI, LV, LT) and bilateral edges weighted by mean cross-border flow magnitudes. Two GATv2Conv layers capture spatial dependencies; a two-layer GRU captures temporal dynamics over a 48-hour input window. The model predicts three quantiles (P10/P50/P90) for Estonian domestic production 24 hours ahead using pinball loss, trained on 2019–Oct 2025 and tested on the held-out January 2026 crisis month.

| | S0: Connected | S1: Isolated | S2: +323 MW wind | S3: +887 MW wind |
|-|---|---|---|---|
| **Cross-border flows** | Actual | Zeroed | Zeroed | Zeroed |
| **Wind added** | — | — | +323 MW (Scenario A) | +887 MW (Scenario B) |
| **Available energy** | Production + actual imports | Production only | Production + wind A | Production + wind B |

In [ ]:
try:
    supply = pd.read_csv(SUPPLY_CSV,      parse_dates=['timestamp'], index_col='timestamp')
    avail  = pd.read_csv(TOTAL_AVAIL_CSV, parse_dates=['timestamp'], index_col='timestamp')
    if supply.index.tz is None: supply.index = supply.index.tz_localize('UTC')
    if avail.index.tz  is None: avail.index  = avail.index.tz_localize('UTC')
    idx = supply.index.to_numpy()

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    # Top-left: GNN domestic production P50 — all scenarios
    ax = axes[0, 0]
    ax.plot(idx, supply['supply_s1_p50'].values, color='green',  lw=2,   label='S0: Connected')
    ax.plot(idx, supply['supply_s2_p50'].values, color='red',    lw=2,   label='S1: Isolated')
    ax.plot(idx, supply['supply_s3_p50'].values, color='orange', lw=1.5, label='S2: +323 MW wind')
    ax.plot(idx, supply['supply_s4_p50'].values, color='gold',   lw=1.5, label='S3: +887 MW wind')
    ax.fill_between(idx, supply['supply_s1_p50'].values, supply['supply_s2_p50'].values,
                    alpha=0.12, color='blue', label='Isolation cost')
    ax.set_title('Domestic production P50 by scenario')
    ax.set_ylabel('Production (MW)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.tick_params(axis='x', rotation=30)

    # Top-right: total available energy (production + imports for S0)
    ax = axes[0, 1]
    ax.fill_between(idx, avail['supply_s0_p10'].values, avail['supply_s0_p90'].values, alpha=0.12, color='green')
    ax.plot(idx, avail['supply_s0_p50'].values,    color='green',   lw=2,   label='S0: Connected (prod+imports)')
    ax.plot(idx, avail['supply_s0iso_p50'].values, color='darkred', lw=2,   linestyle='--', label='S0: Isolated, no response')
    ax.plot(idx, avail['supply_s1_p50'].values,    color='red',     lw=2,   label='S1: Isolated (plants respond)')
    ax.plot(idx, avail['supply_s2_p50'].values,    color='orange',  lw=1.5, label='S2: +323 MW wind')
    ax.plot(idx, avail['supply_s3_p50'].values,    color='gold',    lw=1.5, label='S3: +887 MW wind')
    ax.set_title('Total available energy by scenario')
    ax.set_ylabel('Available energy (MW)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.tick_params(axis='x', rotation=30)

    # Bottom-left: mean production bar (4 scenarios)
    ax = axes[1, 0]
    sc_keys    = ['s1','s2','s3','s4']
    sc_labels  = ['S0\nConnected','S1\nIsolated','S2\n+323 MW','S3\n+887 MW']
    sc_colors  = ['green','red','orange','gold']
    p50m = [supply[f'supply_{k}_p50'].mean() for k in sc_keys]
    p10m = [supply[f'supply_{k}_p10'].mean() for k in sc_keys]
    p90m = [supply[f'supply_{k}_p90'].mean() for k in sc_keys]
    err_lo = [max(0, p50-p10) for p50,p10 in zip(p50m,p10m)]
    err_hi = [max(0, p90-p50) for p50,p90 in zip(p50m,p90m)]
    x4 = np.arange(4)
    bars = ax.bar(x4, p50m, 0.5, color=sc_colors, alpha=0.8)
    ax.errorbar(x4, p50m, yerr=[err_lo,err_hi], fmt='none', color='black', capsize=6, lw=1.5)
    for bar, val in zip(bars, p50m):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8, f'{val:.0f}', ha='center', fontsize=8)
    ax.axhline(900, color='black', lw=1.5, linestyle='--', label='~900 MW consumption')
    ax.set_xticks(x4); ax.set_xticklabels(sc_labels, fontsize=9)
    ax.set_ylabel('Mean production (MW)')
    ax.set_title('Mean domestic production — P10/P90 error bars')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

    # Bottom-right: mean total available energy (5 scenarios)
    ax = axes[1, 1]
    av_p50k = ['supply_s0_p50','supply_s0iso_p50','supply_s1_p50','supply_s2_p50','supply_s3_p50']
    av_p10k = ['supply_s0_p10','supply_s0iso_p10','supply_s1_p10','supply_s2_p10','supply_s3_p10']
    av_p90k = ['supply_s0_p90','supply_s0iso_p90','supply_s1_p90','supply_s2_p90','supply_s3_p90']
    av_lbls = ['S0\nConnected','S0\nIso\n(no resp.)','S1\nIsolated','S2\n+323 MW','S3\n+887 MW']
    av_cols = ['green','darkred','red','orange','gold']
    av_p50 = [avail[k].mean() for k in av_p50k]
    av_p10 = [avail[k].mean() for k in av_p10k]
    av_p90 = [avail[k].mean() for k in av_p90k]
    av_lo  = [max(0, p50-p10) for p50,p10 in zip(av_p50,av_p10)]
    av_hi  = [max(0, p90-p50) for p50,p90 in zip(av_p50,av_p90)]
    x5 = np.arange(5)
    bars = ax.bar(x5, av_p50, 0.5, color=av_cols, alpha=0.8)
    ax.errorbar(x5, av_p50, yerr=[av_lo,av_hi], fmt='none', color='black', capsize=6, lw=1.5)
    for bar, val in zip(bars, av_p50):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8, f'{val:.0f}', ha='center', fontsize=8)
    ax.axhline(900, color='black', lw=1.5, linestyle='--', label='~900 MW consumption')
    ax.set_xticks(x5); ax.set_xticklabels(av_lbls, fontsize=8)
    ax.set_ylabel('Mean available energy (MW)')
    ax.set_title('Mean total available energy\n(S0 connected = production + actual imports)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

    plt.suptitle('GNN Supply Scenarios — January 2026', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/final_gnn_scenarios.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Mean available energy (P50):')
    for lbl, val in zip(av_lbls, av_p50):
        print(f"  {lbl.replace(chr(10),' '):22s}: {val:.0f} MW")
except FileNotFoundError as e:
    print(f'Missing: {e}\nRun Supply_wind.py then available_energy_analysis.ipynb')

---
## Resilience Simulation

Supply and demand are combined to compute hourly surplus and deficit probabilities. The **typical case** compares GNN P50 supply vs SARIMAX P50 demand. The **stress test** compares P10 supply vs P95 demand — worst-case supply against highest expected consumption. Deficit hours are counted when surplus < 0.

In [ ]:
try:
    avail  = pd.read_csv(TOTAL_AVAIL_CSV, parse_dates=['timestamp'], index_col='timestamp')
    demand = pd.read_csv(DEMAND_CSV,      parse_dates=['timestamp'], index_col='timestamp')
    if avail.index.tz  is None: avail.index  = avail.index.tz_localize('UTC')
    if demand.index.tz is None: demand.index = demand.index.tz_localize('UTC')
    common = avail.index.intersection(demand.index)
    av  = avail.loc[common]
    dem = demand.loc[common]
    idx = common.to_numpy()
    d50 = dem['demand_p50'].values
    d95 = dem['demand_p95'].values

    scenarios = [
        ('supply_s0',     'S0: Connected',          'green'),
        ('supply_s0iso',  'S0: Isolated (no resp.)', 'darkred'),
        ('supply_s1',     'S1: Isolated',            'red'),
        ('supply_s2',     'S2: +323 MW wind',        'orange'),
        ('supply_s3',     'S3: +887 MW wind',        'gold'),
    ]

    print(f"{'Scenario':<28} {'Mean surplus':>13} {'Def hrs (typ)':>14} {'Def hrs (stress)':>17}")
    print('-' * 76)
    rows = []
    for key, label, color in scenarios:
        s50 = av[f'{key}_p50'].values
        s10 = av[f'{key}_p10'].values
        s_typ    = s50 - d50
        s_stress = s10 - d95
        dt = int((s_typ < 0).sum())
        ds = int((s_stress < 0).sum())
        print(f'{label:<28} {s_typ.mean():>10.0f} MW {dt:>14} {ds:>17}')
        rows.append((label, color, s_typ, s_stress, dt, ds))

    fig, axes = plt.subplots(3, 1, figsize=(16, 12))

    for label, color, s_typ, _, _, _ in rows:
        axes[0].plot(idx, s_typ, color=color, lw=1.5, label=label)
    axes[0].axhline(0, color='black', lw=1.5, linestyle='--')
    axes[0].fill_between(idx, 0, -3000, alpha=0.04, color='red')
    axes[0].set_ylabel('Surplus (MW)')
    axes[0].set_title('Typical: supply P50 − demand P50')
    axes[0].legend(fontsize=8, loc='upper right'); axes[0].grid(alpha=0.3)

    for label, color, _, s_stress, _, _ in rows:
        axes[1].plot(idx, s_stress, color=color, lw=1.5, label=label)
    axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
    axes[1].fill_between(idx, 0, -3000, alpha=0.04, color='red')
    axes[1].set_ylabel('Surplus (MW)')
    axes[1].set_title('Stress test: supply P10 − demand P95')
    axes[1].legend(fontsize=8, loc='upper right'); axes[1].grid(alpha=0.3)
    axes[1].tick_params(axis='x', rotation=30)

    x = np.arange(len(rows)); w = 0.35
    bars1 = axes[2].bar(x - w/2, [r[4] for r in rows], w,
                        color=[r[1] for r in rows], alpha=0.85, label='Typical (P50 vs P50)')
    bars2 = axes[2].bar(x + w/2, [r[5] for r in rows], w,
                        color=[r[1] for r in rows], alpha=0.4, hatch='//', label='Stress (P10 vs P95)')
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        axes[2].text(bar.get_x()+bar.get_width()/2, h+5, str(int(h)), ha='center', fontsize=8)
    axes[2].set_xticks(x); axes[2].set_xticklabels([r[0] for r in rows], fontsize=9)
    axes[2].set_ylabel('Deficit hours (out of 744)')
    axes[2].set_title('Deficit hours by scenario')
    axes[2].axhline(744, color='gray', lw=1, linestyle=':', alpha=0.5)
    axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3, axis='y')

    plt.suptitle('Resilience Analysis — January 2026', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/final_resilience.png', dpi=150, bbox_inches='tight')
    plt.show()
except FileNotFoundError as e:
    print(f'Missing: {e}\nRun available_energy_analysis.ipynb and Timeseries.ipynb first')

---
## Conclusion

**Import dependency is large.** Cross-border flows provided ~600 MW on average in January 2026 — around 60% of hourly consumption. Full isolation without any production response would leave Estonia in deficit almost every hour.

**Power plants respond, but the gap is large.** The GNN's isolation scenario shows domestic production rising slightly when flows are zeroed, reflecting historical dispatch behavior. But the step from S0 isolated (no response) to S1 isolated (plants respond) is modest — dispatchable thermal capacity alone cannot bridge the import gap.

**Wind capacity materially reduces deficit hours.** Both Scenario A (+323 MW) and Scenario B (+887 MW) improve the isolated energy balance. The impact depends critically on weather: January 2026 capacity factors (~15%) were roughly half the historical January average (~30%), so even large installed capacity generated relatively little energy during the crisis.

**Wind alone is not sufficient.** Full resilience during an extreme isolation + low-wind event requires wind expansion paired with strategic reserves (gas/hydro) and demand flexibility. Scenario B with supplementary storage and dispatchable backup gets Estonia close to self-sufficiency in normal isolation conditions; Scenario A alone does not.

**Limitations.** The model uses a single weather point, assumes perfect plant dispatch foresight, and does not model demand-side response or transmission losses. Residual SARIMAX autocorrelation means the Monte Carlo demand band is slightly too narrow. These are acknowledged limitations for future work.